In [7]:
import os
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import sys 
sys.path.append(os.path.join(os.getcwd(), "..", "src", "ingestion"))
from chunker import chunker_text

In [8]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print("model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

model loaded


In [9]:
def build_index(chunk_size, overlap, folder_path="../data/raw_docs"):
    all_chunks = []

    for file_name in os.listdir(folder_path):
        if file_name.endswith(".txt"):
            file_path = os.path.join(folder_path, file_name)
            with open(file_path, "r", encoding="utf-8") as f:
                text = f.read()

            chunks = chunker_text(text, chunk_size, overlap)
            for i, chunk in enumerate(chunks):
                all_chunks.append({
                    "text": chunk,
                    "source": file_name,
                    "chunk_id": i,
                })

    texts = [c["text"] for c in all_chunks]
    embeddings = model.encode(texts)
    embeddings = np.array(embeddings).astype("float32")

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    return index, all_chunks

In [10]:
indices = {
    150: build_index(chunk_size=150, overlap=30),
    300: build_index(chunk_size=300, overlap=50),
    500: build_index(chunk_size=500, overlap=80),
}

print("Built indices for chunk sizes:", list(indices.keys()))

Built indices for chunk sizes: [150, 300, 500]


In [11]:
print("Sample source_doc from questions.csv:", questions_df["source_doc"].iloc[0])
print("Sample source from chunk metadata:", indices[150][1][0]["source"])

Sample source_doc from questions.csv: Turing_test.txt
Sample source from chunk metadata: AI_winter.txt


In [12]:
sources_in_index = set(c["source"] for c in indices[150][1])
print(sources_in_index)
print("Turing_test.txt" in sources_in_index)

{'AlexNet.txt', 'AI_winter.txt', 'Expert_system.txt', 'Machine_learning.txt', 'Perceptron.txt', 'Transformer_(deep_learning_architecture).txt', 'Timeline_of_artificial_intelligence.txt', 'ImageNet.txt', 'Deep_Blue_(chess_computer).txt', 'History_of_artificial_intelligence.txt', 'Turing_test.txt'}
True


In [13]:
def hit_rate(question, source_doc, chunk_size, top_k):
    index, chunk_metadata = indices[chunk_size]

    query_embedding = model.encode([question])
    query_embedding = np.array(query_embedding).astype("float32")
    distances, retrived_indices = index.search(query_embedding, top_k)

    retrived_sources = [chunk_metadata[i]["source"] for i in retrived_indices[0]]
    
    correct = sum(1 for s in retrived_sources if s == source_doc)

    return correct / top_k

In [14]:
questions_df = pd.read_csv("../data/questions.csv")

chunk_sizes = [150, 300, 500]
top_ks = [3, 5]

results = []

for _, row in questions_df.iterrows():
    question = row["question"]
    source_doc = row["source_doc"]

    best_score = -1
    best_chunk_size = None
    best_top_k = None

    for cs in chunk_sizes:
        for tk in top_ks:
            score = hit_rate(question, source_doc, cs, tk)

            if score > best_score or (
                score == best_score and
                (best_chunk_size is None or cs < best_chunk_size or (cs == best_chunk_size and tk < best_top_k))
            ):
                best_score = score
                best_chunk_size = cs
                best_top_k = tk

    results.append({
        "question": question,
        "question_type": row["question_type"],
        "source_doc": source_doc,
        "best_chunk_size": best_chunk_size,
        "best_top_k": best_top_k,
        "best_hit_rate": best_score,
    })

print(f"Processed {len(results)} questions")

Processed 58 questions


In [15]:
labled_df = pd.DataFrame(results)
labled_df.to_csv("../data/labeled_questions.csv", index=False)
labled_df.head()

,question,question_type,source_doc,best_chunk_size,best_top_k,best_hit_rate
0,When was the Turing Test first proposed?,factual,Turing_test.txt,150,5,0.800000
1,What did Alan Turing propose as a way to test ...,definitional,Turing_test.txt,150,5,0.800000
2,What is a perceptron and who invented it?,definitional,Perceptron.txt,150,3,1.000000
3,Why did early perceptrons fail to solve certai...,factual,Perceptron.txt,300,3,0.666667
4,What caused the first AI winter?,factual,AI_winter.txt,150,3,0.666667


In [16]:
print(labled_df["best_chunk_size"].value_counts())
print()
print(labled_df["best_top_k"].value_counts())
print()
print("average hit rate:", labled_df["best_hit_rate"].mean())


best_chunk_size
150    49
500     5
300     4
Name: count, dtype: int64

best_top_k
3    50
5     8
Name: count, dtype: int64

average hit rate: 0.5149425287356322
